In [13]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM



model_name = "gpt2-medium"
model=AutoModelForCausalLM.from_pretrained(model_name)
tokenise=AutoTokenizer.from_pretrained(model_name)
model.eval()
device=torch.device('cuda'if torch.cuda.is_available() else'cpu')
model.to(device)


prompt = f"Question:{'How do I make a...?'} \nAnswer:"

inputs=tokenise(prompt,return_tensors='pt').to(model.device)
input_ids=inputs["input_ids"].to(model.device)
with torch.no_grad():
  outputs=model(**inputs)
# extract logits at last position
logits=outputs.logits
next_logits=logits[:,-1,:]

prob= F.softmax(next_logits,dim=-1)
top_k= torch.topk(prob,dim=-1,k=50)
indices=top_k.indices.squeeze(0)
values=top_k.values.squeeze(0)
print(top_k.indices.shape)
for i in range(50):
    token=tokenise.decode([indices[i].item()])
    proba=values[i]
    print(f"Token: '{token}'| Probability{proba.item():.4f}")
    
# implentation of TransformerLens
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2")

def get_topk_tokens(probs, k=50):
    topk = torch.topk(probs, k=k)
    return topk.indices, topk.values

def separate_harmful(topk_ids, topk_probs, harmful_ids):
    # TODO: split topk into harmful vs benign
    pass

def separation_score(harmful_probs, benign_probs):
    # TODO: mean difference
    pass

def ablate_layer(model, prompt, layer_idx):
    # TODO: use model.run_with_hooks() to zero out layer_idx
    pass

def run(prompt, harmful_tokens, k=50):
    tokens = model.to_tokens(prompt)
    harmful_ids = [model.to_single_token(t) for t in harmful_tokens]

    logits, cache = model.run_with_cache(tokens)
    probs = torch.softmax(logits[0, -1, :], dim=-1)

    topk_ids, topk_probs = get_topk_tokens(probs, k)
    harmful_probs, benign_probs = separate_harmful(topk_ids, topk_probs, harmful_ids)
    score = separation_score(harmful_probs, benign_probs)

    # TODO: loop over layers, ablate each, recompute score, compare
    
    return score, cache

if __name__ == "__main__":
    prompt = "TODO"
    harmful_tokens = ["TODO"]
    score, cache = run(prompt, harmful_tokens)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 50])
Token: 'You'| Probability0.0842
Token: 'The'| Probability0.0345
Token: 'To'| Probability0.0301
Token: 'There'| Probability0.0270
Token: 'Go'| Probability0.0260
Token: 'I'| Probability0.0248
Token: 'First'| Probability0.0245
Token: 'This'| Probability0.0227
Token: 'If'| Probability0.0206
Token: 'Make'| Probability0.0197
Token: '1'| Probability0.0177
Token: '
'| Probability0.0168
Token: 'By'| Probability0.0147
Token: 'In'| Probability0.0140
Token: 'It'| Probability0.0133
Token: 'Just'| Probability0.0120
Token: 'Use'| Probability0.0107
Token: 'Well'| Probability0.0105
Token: 'When'| Probability0.0097
Token: 'A'| Probability0.0081
Token: 'Take'| Probability0.0080
Token: 'Please'| Probability0.0078
Token: 'For'| Probability0.0077
Token: ' You'| Probability0.0075
Token: 'Simply'| Probability0.0073
Token: 'We'| Probability0.0061
Token: 'How'| Probability0.0060
Token: 'As'| Probability0.0056
Token: 'Here'| Probability0.0056
Token: 'From'| Probability0.0052
Token: 'Press'| P